# 138 - Jina CLIP v2 調査総合まとめ

NB131-136 の結果を集約し、**置き換える／置き換えない**の判断材料を一枚に整理する。

## 調査範囲
- NB131: 画像埋め込み品質（t-SNE / シルエット / Trustworthiness）
- NB132: Text→Image MRR（EN/JA 18 クエリ）
- NB133: ベクトル空間特性（モダリティギャップ / 異方性 / Matryoshka）
- NB134: クエリ工学（プロンプトプレフィクス / 言い換え / クロス言語）
- NB135: ONNX 量子化（fp32/fp16/int8/q4/q4f16）
- NB136: JS (Transformers.js) 互換性

## 結論（見出し）
**置き換え見送り推奨**。ただし (1) 多言語化 (2) 超軽量エッジ配布 (3) 100万枚超スケール の要件化時は **Jina CLIP v2 + 32-D Matryoshka + q4f16** が有力候補。

In [1]:
import json
from pathlib import Path
import pandas as pd

EVAL = Path("../data/evaluations")
files = {
    "131": "jinaai_jina-clip-v2_2026-04-14.json",
    "132": "132_jina_clip_v2_text_search_2026-04-14.json",
    "133": "133_jina_clip_v2_vector_space_2026-04-14.json",
    "134": "134_jina_clip_v2_query_engineering_2026-04-14.json",
    "135": "135_jina_clip_v2_quantization_2026-04-14.json",
    "136": "136_jina_clip_v2_js_compatibility_2026-04-14.json",
}
data = {k: json.loads((EVAL / v).read_text()) for k, v in files.items()}
print("Loaded evaluation data for NB131-136")

Loaded evaluation data for NB131-136


## 1. Jina CLIP v2 vs SigLIP 2 Large 256: 直接対決表

In [2]:
headline = pd.DataFrame([
    {"指標": "埋め込み次元", "Jina v2": "1024", "SigLIP 2 L256": "1024", "勝者": "同"},
    {"指標": "入力解像度", "Jina v2": "512×512", "SigLIP 2 L256": "256×256", "勝者": "Jina（情報量）"},
    {"指標": "トークナイザ", "Jina v2": "XLM-R BPE", "SigLIP 2 L256": "Gemma BPE", "勝者": "ほぼ互角"},
    {"指標": "対応言語", "Jina v2": "89", "SigLIP 2 L256": "109", "勝者": "SigLIP2"},
    {"指標": "PyTorch (transformers 5.x)", "Jina v2": "❌ 壊れてる", "SigLIP 2 L256": "✅ 動く", "勝者": "SigLIP2"},
    {"指標": "ONNX 公式", "Jina v2": "✅ fp32-q4f16", "SigLIP 2 L256": "✅ fp32-q8", "勝者": "同"},
    {"指標": "Transformers.js", "Jina v2": "✅ JinaCLIPModel", "SigLIP 2 L256": "✅ Siglip*Model", "勝者": "同"},
    {"指標": "Matryoshka 対応", "Jina v2": "✅ 32-D まで耐える", "SigLIP 2 L256": "❌ 256-D で崩壊", "勝者": "Jina"},
    {"指標": "MRR overall (1024-D)", "Jina v2": "0.881", "SigLIP 2 L256": "0.928", "勝者": "SigLIP2"},
    {"指標": "MRR overall (+Query: prefix)", "Jina v2": "0.951", "SigLIP 2 L256": "?", "勝者": "要検証"},
    {"指標": "MRR EN", "Jina v2": "0.896 (+prefix 1.000)", "SigLIP 2 L256": "0.892", "勝者": "Jina"},
    {"指標": "MRR JA", "Jina v2": "0.852", "SigLIP 2 L256": "1.000", "勝者": "SigLIP2"},
    {"指標": "処理速度 fp32 (img/s)", "Jina v2": "6.6", "SigLIP 2 L256": "~4", "勝者": "Jina"},
    {"指標": "処理速度 fp16 (img/s)", "Jina v2": "7.5", "SigLIP 2 L256": "~5", "勝者": "Jina"},
    {"指標": "最小サイズ品質保持", "Jina v2": "32-D + q4f16 → 86MB", "SigLIP 2 L256": "256-D + q8 → 250MB", "勝者": "Jina"},
    {"指標": "JS テキスト完全再現", "Jina v2": "cos 0.999998", "SigLIP 2 L256": "cos ≈ 1", "勝者": "同"},
])
from IPython.display import display
display(headline)

,指標,Jina v2,SigLIP 2 L256,勝者
0,埋め込み次元,1024,1024,同
1,入力解像度,512×512,256×256,Jina（情報量）
2,トークナイザ,XLM-R BPE,Gemma BPE,ほぼ互角
3,対応言語,89,109,SigLIP2
4,PyTorch (transformers 5.x),❌ 壊れてる,✅ 動く,SigLIP2
5,ONNX 公式,✅ fp32-q4f16,✅ fp32-q8,同
6,Transformers.js,✅ JinaCLIPModel,✅ Siglip*Model,同
7,Matryoshka 対応,✅ 32-D まで耐える,❌ 256-D で崩壊,Jina
8,MRR overall (1024-D),0.881,0.928,SigLIP2
9,MRR overall (+Query: prefix),0.951,?,要検証


## 2. 主要数値の再掲

In [3]:
print("=== NB131: 画像埋め込み品質 ===")
print(f"  Silhouette(2D): Jina 0.170 / CLIP-L 0.221 / SigLIP2 Base 0.156")
print(f"  Trustworthiness(2D): Jina 0.974 / SigLIP2 Base 0.958 — Jina best")
print(f"  距離比(3D): Jina 0.550 / 他 0.62-0.65 — Jina best")

print("\n=== NB132: Text→Image MRR ===")
for row in data['132']['per_query'][:5]:
    print(f"  {row['query'][:40]:40s} MRR={row['MRR']:.2f}")

print("\n=== NB133: Matryoshka MRR ===")
m = pd.DataFrame(data['133']['matryoshka'])
print(m.pivot(index='dim', columns='model', values='MRR_overall').sort_index(ascending=False).round(3).to_string())

print("\n=== NB134: プロンプトプレフィクス ===")
for lang, d in data['134']['prompt_template_summary'].items():
    best = max(d.items(), key=lambda kv: kv[1])
    print(f"  {lang}: best template = {best[0]} (MRR={best[1]:.4f})")

print("\n=== NB135: 量子化 ===")
for row in data['135']['summary']:
    print(f"  {row['dtype']:<8} size={row['size_MB']:>7.0f}MB  speed={row['img/sec']:>5.2f} img/s  MRR={row['MRR_overall']:.4f}")

print("\n=== NB136: JS 互換性 ===")
d = data['136']
print(f"  Text cos (Python vs JS): mean={d['txt_cos']['mean']:.6f}, min={d['txt_cos']['min']:.6f}")
print(f"  Image cos (Python vs JS): mean={d['img_cos']['mean']:.6f}, min={d['img_cos']['min']:.6f}")
print(f"  JS image speed (CPU, q4f16): {d['js_image_speed_per_sec']:.2f} img/s")

=== NB131: 画像埋め込み品質 ===
  Silhouette(2D): Jina 0.170 / CLIP-L 0.221 / SigLIP2 Base 0.156
  Trustworthiness(2D): Jina 0.974 / SigLIP2 Base 0.958 — Jina best
  距離比(3D): Jina 0.550 / 他 0.62-0.65 — Jina best

=== NB132: Text→Image MRR ===
  a park with trees and nature             MRR=1.00
  village park with greenery               MRR=0.25
  night cityscape Tokyo                    MRR=1.00
  city lights at night                     MRR=1.00
  Python conference presentation           MRR=1.00

=== NB133: Matryoshka MRR ===
model  Jina v2  SigLIP 2 L256
dim                          
1024     0.881          0.928
768      0.853          0.854
512      0.881          0.935
384      0.858          0.852
256      0.859          0.853
192      0.914          0.763
128      0.786          0.687
96       0.863          0.688
64       0.881          0.651
48       0.956          0.493
32       0.972          0.475

=== NB134: プロンプトプレフィクス ===
  en: best template = query_prefix (MRR=1.0000)
  ja: be

## 3. 採用判断マトリクス

In [4]:
decision = pd.DataFrame([
    {"要件シナリオ": "JA メインの本プロジェクト現状",
     "推奨": "SigLIP 2 L256 維持", "理由": "JA MRR 1.000 vs Jina 0.852"},
    {"要件シナリオ": "多言語化 (韓中仏など追加)",
     "推奨": "Jina v2 + Query: prefix", "理由": "XLM-R BPE の多言語広さ"},
    {"要件シナリオ": "ブラウザ/エッジ配布",
     "推奨": "Jina v2 q4f16", "理由": "861MB, Text完全再現, 32-D可"},
    {"要件シナリオ": "100 万枚 × 1024-D がストレージ重い",
     "推奨": "Jina v2 + 32-D Matryoshka", "理由": "ストレージ 1/32, MRR 維持(小サンプル)"},
    {"要件シナリオ": "GPU で int8 量子化したい",
     "推奨": "❌ 両モデルとも不可", "理由": "int8 は ORT-CUDA 非最適化, 1450ms/query"},
    {"要件シナリオ": "モダリティ mixed 検索 (img+text 混合)",
     "推奨": "Jina v2", "理由": "モダリティギャップ 20% 小 (cos_dist 0.73 vs 0.95)"},
])
display(decision)

,要件シナリオ,推奨,理由
0,JA メインの本プロジェクト現状,SigLIP 2 L256 維持,JA MRR 1.000 vs Jina 0.852
1,多言語化 (韓中仏など追加),Jina v2 + Query: prefix,XLM-R BPE の多言語広さ
2,ブラウザ/エッジ配布,Jina v2 q4f16,"861MB, Text完全再現, 32-D可"
3,100 万枚 × 1024-D がストレージ重い,Jina v2 + 32-D Matryoshka,"ストレージ 1/32, MRR 維持(小サンプル)"
4,GPU で int8 量子化したい,❌ 両モデルとも不可,"int8 は ORT-CUDA 非最適化, 1450ms/query"
5,モダリティ mixed 検索 (img+text 混合),Jina v2,モダリティギャップ 20% 小 (cos_dist 0.73 vs 0.95)


## 4. 実用パイプライン（Jina v2 採用時）

```
クエリ(EN): "park with trees" → `Query: park with trees` にプレフィクス化
クエリ(JA): "緑の木々" → そのまま (or `森の中の木々` 等にリライト)
                    ↓
            Jina v2 q4f16 (Python/JS 両対応)
                    ↓
            1024-D embedding
                    ↓
       [A] 高品質用: そのまま 1024-D で検索 (MRR 0.95)
       [B] 軽量用:   先頭 32 次元に truncate + 再正規化 (MRR 0.97, 容量 1/32)
                    ↓
              DuckDB FLOAT[1024] or FLOAT[32]
                    ↓
              HNSW cosine 検索
```

## 5. 運用面の注意
- **PyTorch 経路は使わない**: ONNX 一本で Python/JS/ブラウザ統一
- **プロンプトプレフィクス**: EN は `Query: `, JA は素。アプリ側で自動付与
- **q4f16 が最適**: fp16 より 50% 小さく速度同等、MRR 0.905 で十分
- **JA クエリ品質ガード**: `緑の木々` のような曖昧短クエリは `森の中の木々` 等に書き換える UI
- **HF remote code 依存なし**: JS は Transformers.js ネイティブ、Python は JinaCLIPONNXEmbedder（本 PoC 実装）


## 6. 今後の TODO（要件化したら）

- [ ] 多言語化（韓中仏）で同じ 18 クエリを翻訳して MRR 再測
- [ ] 100万枚規模コーパスで 32-D Matryoshka の MRR 再検証（現状 378 枚のみ）
- [ ] SigLIP 2 にも `Query: ` プレフィクスを適用した公平比較（NB132 リフレッシュ）
- [ ] Grounding DINO との ensemble スコア正規化（NB87 拡張、Jina スコアレンジが違う可能性）
- [ ] HNSW（DuckDB vss）で 32-D Jina vs 1024-D SigLIP2 の検索レイテンシ実測
